In [1]:
from itertools import combinations
from joblib import Parallel, delayed, dump, load
import json
from multiprocessing import Pool
import numpy as np
import os
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.decomposition import IncrementalPCA, PCA
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import time
import torch

In [ ]:
NUM_PROCESSES = 8
NUM_FEATURES_PCA = 100

Embeddings

In [6]:
def load_embeddings(file_path):
    key = file_path.split("_")[1][:-4]
    if "video" in file_path:
        try:
            embeddings_type = "video"
            embedding = torch.load(file_path, map_location=torch.device('cpu')).to('cpu').view(32, 4096).detach().numpy()
        except:
            print("Video")
            return None
    elif "audio" in file_path:
        try:
            embeddings_type = "audio"
            embedding = torch.load(file_path, map_location=torch.device('cpu')).to('cpu').view(8, 4096).detach().numpy()
        except:
            return None
    else:
        return None
    return key, embeddings_type, embedding

In [ ]:
videoembeddings = {}
audioembeddings = {}

embs_directory = "embs"
emb_files = [os.path.join(embs_directory, emb) for emb in os.listdir(embs_directory)]
emb_files = set(emb_files)

with Pool(processes=NUM_PROCESSES) as pool:
    results = pool.map(load_embeddings, emb_files)

for result in results:
    if result:
        key, emb_type, embedding = result
        if emb_type == "video":
            videoembeddings[key] = embedding
        elif emb_type == "audio":
            audioembeddings[key] = embedding

Remaining Files: 2639
Video
Video
Video
Video
Video
Video
Video
Video
Video
Video
Video
Video
Video
Video

Video
Video

Creating num_videos x 40*4096 matrix

In [8]:
keys = videoembeddings.keys()
k = np.array(list(keys))

In [9]:
allembeddings = {}
for key in keys:
    try:
        allembeddings[key] = np.concatenate((videoembeddings[key], audioembeddings[key]), axis=0)
    except:
        keys = allembeddings.keys()

In [10]:
k = np.array(list(keys))

In [11]:
num_videos = len(keys)
print("Number of Videos", num_videos)

Number of Videos 0


In [12]:
if num_videos > 0:
    row_tensors = [allembeddings[i].reshape(1, 40 * 4096) for i in keys]
    data_matrix = np.concatenate(row_tensors, axis=0)

Normalization

In [ ]:
if num_videos > 0:
    scaler = MinMaxScaler()
    scaled_features = scaler.fit(data_matrix)
    scaled_features = scaler.transform(data_matrix)

PCA

In [ ]:
if num_videos > 0:
    pca = PCA(n_components=NUM_FEATURES_PCA, random_state=22)
    pca.fit(scaled_features)
    input_matrix = pca.transform(scaled_features)

In [17]:
if num_videos > 0:
    pca_embs_new = {}
    for i in range(len(k)):
        pca_embs_new[k[i]] = list(input_matrix[i, :])